# Submissão 3 — Grupo 7 | MIA | Aprendizagem Profunda
## Notebook B — TF-IDF + SVM + Logistic Regression Ensemble

Classifica 150 textos em 5 classes: **Human, Anthropic, Google, Meta, OpenAI**.

### Abordagem
- **Pré-processamento**: TF-IDF (unigramas + bigramas) + features estilísticas
- **Modelo 1**: LinearSVC — excelente para text classification com poucos dados
- **Modelo 2**: Logistic Regression com regularização L2
- **Ensemble**: média ponderada das probabilidades calibradas
- **Output**: `subm3-g7-MIA-B.csv` com colunas ID;Label

In [9]:
# ── Instalação de dependências ──────────────────────────────────────────────
import subprocess, sys
for pkg in ['scikit-learn', 'pandas', 'numpy']:
    try:
        __import__(pkg.replace('-','_'))
        print(f'  {pkg} ok')
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f'  {pkg} instalado')
print('Dependências prontas.')

  scikit-learn instalado
  pandas ok
  numpy ok
Dependências prontas.


In [10]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from scipy.sparse import hstack, csr_matrix

np.random.seed(42)

# ── Paths — ajusta conforme necessário ────────────────────────────────────
TRAIN_PATH = 'dataset-exemplos.csv'   # dataset do professor
# Se tiveres o dataset maior da Tarefa 1, usa este em vez do anterior:
# TRAIN_PATH = 'train.csv'
TEST_PATH  = 'subm3.csv'
OUT_PATH   = 'subm3-g7-MIA-B.csv'

CLASSES = ['Human', 'Anthropic', 'Google', 'Meta', 'OpenAI']
print('Configuração carregada.')

Configuração carregada.


In [11]:
# ── Carregar dados ─────────────────────────────────────────────────────────
df_train = pd.read_csv(TRAIN_PATH, sep=';', encoding='utf-8-sig')
df_test  = pd.read_csv(TEST_PATH,  sep=';', encoding='utf-8-sig')

print(f'Treino: {len(df_train)} exemplos')
print(f'Teste:  {len(df_test)} exemplos')
print(f'\nDistribuição treino:\n{df_train["Label"].value_counts()}')
print(f'\nColunas teste: {df_test.columns.tolist()}')

Treino: 325 exemplos
Teste:  150 exemplos

Distribuição treino:
Label
Human        120
Anthropic     56
Meta          51
Google        51
OpenAI        47
Name: count, dtype: int64

Colunas teste: ['ID', 'Text']


In [12]:
# ── Features estilísticas ──────────────────────────────────────────────────
def extract_stylistic(texts):
    """8 features que capturam o estilo de escrita."""
    feats = []
    for text in texts:
        words     = text.split()
        sentences = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
        n_words   = max(len(words), 1)
        n_sents   = max(len(sentences), 1)
        feats.append([
            np.mean([len(w) for w in words]) if words else 0,  # avg word length
            n_words / n_sents,                                  # avg sentence length
            len(set(w.lower() for w in words)) / n_words,      # vocabulary richness
            sum(1 for c in text if c in '.,;:!?') / max(len(text), 1),  # punctuation
            text.count(',')  / n_words,                         # comma rate
            (text.count('"') + text.count("'")) / n_words,     # quote rate
            sum(1 for w in words if len(w) > 8) / n_words,     # long word rate
            n_words,                                            # total words
        ])
    return np.array(feats, dtype=np.float32)

print('Função de features estilísticas definida.')

Função de features estilísticas definida.


In [13]:
# ── Preparar features ──────────────────────────────────────────────────────
X_texts_train = df_train['Text'].tolist()
X_texts_test  = df_test['Text'].tolist()
y_train = df_train['Label'].tolist()

# TF-IDF: unigramas + bigramas, máx 15000 features
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=15000,
    sublinear_tf=True,
    strip_accents='unicode',
    analyzer='word',
    min_df=1,
)
X_tfidf_train = tfidf.fit_transform(X_texts_train)
X_tfidf_test  = tfidf.transform(X_texts_test)

# Features estilísticas
sty_train = extract_stylistic(X_texts_train)
sty_test  = extract_stylistic(X_texts_test)
scaler = StandardScaler()
sty_train = scaler.fit_transform(sty_train)
sty_test  = scaler.transform(sty_test)

# Combinar
X_train = hstack([X_tfidf_train, csr_matrix(sty_train)])
X_test  = hstack([X_tfidf_test,  csr_matrix(sty_test)])

print(f'Feature matrix: {X_train.shape[0]} x {X_train.shape[1]}')

Feature matrix: 325 x 15008


In [14]:
# ── Validação cruzada ──────────────────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

svm_base = LinearSVC(C=1.0, max_iter=2000, random_state=42)
svm_cal  = CalibratedClassifierCV(svm_base, cv=3)
lr_model = LogisticRegression(C=5.0, max_iter=1000, random_state=42, solver='lbfgs')

scores_svm = cross_val_score(svm_cal,  X_train, y_train, cv=cv, scoring='accuracy')
scores_lr  = cross_val_score(lr_model, X_train, y_train, cv=cv, scoring='accuracy')

print(f'LinearSVC (calibrado) CV accuracy: {scores_svm.mean():.3f} ± {scores_svm.std():.3f}')
print(f'Logistic Regression   CV accuracy: {scores_lr.mean():.3f} ± {scores_lr.std():.3f}')

LinearSVC (calibrado) CV accuracy: 0.578 ± 0.037
Logistic Regression   CV accuracy: 0.603 ± 0.043


In [15]:
# ── Treinar nos dados completos ────────────────────────────────────────────
print('A treinar SVM...')
svm_cal.fit(X_train, y_train)

print('A treinar Logistic Regression...')
lr_model.fit(X_train, y_train)

# Pesos proporcionais à CV accuracy
w_svm = scores_svm.mean()
w_lr  = scores_lr.mean()
total = w_svm + w_lr

# Ensemble
probs_svm = svm_cal.predict_proba(X_test)
probs_lr  = lr_model.predict_proba(X_test)

# Garantir mesma ordem de classes
def reorder_probs(model, probs, target_classes):
    cls_order = list(model.classes_)
    reordered = np.zeros((probs.shape[0], len(target_classes)))
    for i, cls in enumerate(target_classes):
        if cls in cls_order:
            reordered[:, i] = probs[:, cls_order.index(cls)]
    return reordered

probs_svm_r = reorder_probs(svm_cal,  probs_svm, CLASSES)
probs_lr_r  = reorder_probs(lr_model, probs_lr,  CLASSES)

probs_ens = (w_svm * probs_svm_r + w_lr * probs_lr_r) / total
pred_idx  = probs_ens.argmax(axis=1)
predictions = [CLASSES[i] for i in pred_idx]

print(f'\nDistribuição das previsões:\n{pd.Series(predictions).value_counts()}')

A treinar SVM...
A treinar Logistic Regression...

Distribuição das previsões:
Human        57
Google       30
Meta         29
OpenAI       19
Anthropic    15
Name: count, dtype: int64


In [16]:
# ── Guardar CSV ───────────────────────────────────────────────────────────
# Apenas ID e Label — sem coluna de texto
df_out = pd.DataFrame({'ID': df_test['ID'], 'Label': predictions})
df_out.to_csv(OUT_PATH, sep=';', index=False)

print(f'Guardado: {OUT_PATH}')
print(f'Total: {len(df_out)} previsões')
print(f'\nPrimeiras linhas:')
df_out.head(10)

Guardado: subm3-g7-MIA-B.csv
Total: 150 previsões

Primeiras linhas:


,ID,Label
0,D2-126,Human
1,D2-127,Human
2,D2-128,Anthropic
3,D2-129,Human
4,D2-130,Meta
5,D2-131,Meta
6,D2-132,Human
7,D2-133,Anthropic
8,D2-134,Meta
9,D2-135,Google
